In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC


# 1. WCZYTANIE DANYCH I PODZIAŁ
# Wczytanie pliku CSV
df = pd.read_csv("cases_clinical_for_lab12.csv")

# Rozdzielenie cech (X) i zmiennej docelowej (y)
X = df.drop("high_risk_cvd", axis=1)
y = df["high_risk_cvd"]

# Podział na zbiór treningowy i testowy (stratyfikowany)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 2. PREPROCESSING (Przygotowanie danych)

num_cols =["age", "bmi", "systolic_bp", "diastolic_bp", "glucose"]
cat_cols =["sex", "smoker", "family_history"]

# Pipeline dla cech numerycznych: imputacja medianą i standaryzacja
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline dla cech kategorycznych: imputacja najczęstszą i One-Hot
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Połączenie w jeden ColumnTransformer
preprocess = ColumnTransformer(transformers=[
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols)
])


# 3. TRENOWANIE MODELI (Regresja Logistyczna i SVM)

# Baseline: Regresja Logistyczna
lr_pipeline = Pipeline(steps=[
    ("pre", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
lr_pipeline.fit(X_train, y_train)

#Model SVM z probability=True
svm_pipeline = Pipeline(steps=[
    ("pre", preprocess),
    ("model", SVC(probability=True, random_state=42))
])
svm_pipeline.fit(X_train, y_train)


# 4. OCENA (Progi, Błędy, Niezgodności)

print("TRENOWANIE ZAKOŃCZONE")

# Obliczanie prawdopodobieństw dla obu modeli
proba_lr = lr_pipeline.predict_proba(X_test)[:, 1]
pred_lr = (proba_lr >= 0.5).astype(int)

proba_svm = svm_pipeline.predict_proba(X_test)[:, 1]
pred_svm = (proba_svm >= 0.5).astype(int)

print("\nWpływ progów decyzyjnych (tau) na FN i FP (Model SVM)")
thresholds =[0.4, 0.5, 0.6]
for t in thresholds:
    pred_t = (proba_svm >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_t).ravel()
    print(f"Próg {t:.1f}: False Negatives (FN) = {fn:3d} | False Positives (FP) = {fp:3d}")

print("\nIdentyfikacja przypadków, gdzie SVM i LR się NIE zgadzają")
# Zestawienie wyników na zbiorze testowym
results = X_test.copy()
results['y_true'] = y_test.values
results['pred_lr'] = pred_lr
results['pred_svm'] = pred_svm
results['proba_svm'] = np.round(proba_svm, 3)

# Oznaczanie błędów SVM (żeby było łatwiej wybrać do raportu)
results['error_svm'] =["FN" if (y_t == 1 and p == 0) else ("FP" if (y_t == 0 and p == 1) else "OK")
                        for y_t, p in zip(results['y_true'], results['pred_svm'])]

# Znajdujemy różnice między LR a SVM
disagreements = results[results['pred_lr'] != results['pred_svm']]
print(f"Liczba niezgodności między LR a SVM: {len(disagreements)}")
if len(disagreements) > 0:
    display(disagreements.head(3))

print("\nBŁĘDY DO ANALIZY W RAPORCIE (Kliniczne Case Studies)")
print("Przykładowe 3 przypadki False Positives (FP) dla SVM (Model mówi 'Chory', a pacjent jest 'Zdrowy'):")
display(results[results['error_svm'] == 'FP'].head(3))

print("\nPrzykładowe 3 przypadki False Negatives (FN) dla SVM (Model mówi 'Zdrowy', a pacjent jest 'Chory'):")
display(results[results['error_svm'] == 'FN'].head(3))

TRENOWANIE ZAKOŃCZONE

Wpływ progów decyzyjnych (tau) na FN i FP (Model SVM)
Próg 0.4: False Negatives (FN) =  15 | False Positives (FP) =  76
Próg 0.5: False Negatives (FN) =  24 | False Positives (FP) =  63
Próg 0.6: False Negatives (FN) =  35 | False Positives (FP) =  44

Identyfikacja przypadków, gdzie SVM i LR się NIE zgadzają
Liczba niezgodności między LR a SVM: 23


,sex,age,bmi,systolic_bp,diastolic_bp,glucose,smoker,family_history,y_true,pred_lr,pred_svm,proba_svm,error_svm
248,F,29,30.5,137.0,81,83.0,no,yes,0,1,0,0.453,OK
129,M,41,23.5,141.0,75,90.0,no,no,0,0,1,0.547,FP
265,F,24,25.7,124.0,99,125.0,no,no,1,0,1,0.692,OK



BŁĘDY DO ANALIZY W RAPORCIE (Kliniczne Case Studies)
Przykładowe 3 przypadki False Positives (FP) dla SVM (Model mówi 'Chory', a pacjent jest 'Zdrowy'):


,sex,age,bmi,systolic_bp,diastolic_bp,glucose,smoker,family_history,y_true,pred_lr,pred_svm,proba_svm,error_svm
532,F,84,28.2,130.0,79,88.0,no,no,0,1,1,0.804,FP
160,F,76,29.7,119.0,87,NaN,yes,yes,0,1,1,0.819,FP
685,F,52,NaN,136.0,60,102.0,no,yes,0,1,1,0.614,FP



Przykładowe 3 przypadki False Negatives (FN) dla SVM (Model mówi 'Zdrowy', a pacjent jest 'Chory'):


,sex,age,bmi,systolic_bp,diastolic_bp,glucose,smoker,family_history,y_true,pred_lr,pred_svm,proba_svm,error_svm
726,F,27,24.7,112.0,87,114.0,no,no,1,0,0,0.493,FN
975,M,51,21.2,138.0,92,84.0,no,no,1,0,0,0.416,FN
393,M,33,28.9,136.0,78,62.0,no,no,1,0,0,0.380,FN
